Etapa 1 — Instalação das dependências/Montagem do Drive

Objetivo

Nesta etapa instalamos todas as bibliotecas necessárias para o projeto. Elas serão utilizadas para carregar o modelo FLAN-T5, manipular o dataset, realizar o fine-tuning e avaliar os resultados.

Execute esta célula apenas uma vez no início do notebook.

In [1]:
# ==========================================================
# ETAPA 1 - Instalação das dependências
# ==========================================================
!pip install rouge-score
!pip install -q \
    transformers \
    datasets \
    accelerate \
    sentencepiece \
    evaluate

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Etapa 2 — Configuração do projeto
Objetivo

Nesta etapa definimos as pastas e importamos as bibliotecas que serão utilizadas durante o projeto. Todos os arquivos (dataset, modelo treinado e resultados) serão armazenados no Google Drive.

In [2]:
# ==========================================================
# ETAPA 2 - Configuração do Projeto
# ==========================================================

# -----------------------------
# Importação das bibliotecas
# -----------------------------

from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer
)

# -----------------------------
# Configurações gerais
# -----------------------------

MODEL_NAME = "google/flan-t5-base"
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -----------------------------
# Estrutura do projeto
# -----------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/SubTechChallenge3")

DATASET_DIR = PROJECT_DIR / "dataset"
METADATA_DIR = DATASET_DIR / "metadata"
PDF_PARSES_DIR = DATASET_DIR / "pdf_parses"

MODEL_DIR = PROJECT_DIR / "model"
OUTPUT_DIR = PROJECT_DIR / "output"

for folder in [
    PROJECT_DIR,
    DATASET_DIR,
    METADATA_DIR,
    PDF_PARSES_DIR,
    MODEL_DIR,
    OUTPUT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Arquivos do dataset
# -----------------------------

METADATA_FILE = METADATA_DIR / "sample.jsonl"
PDF_PARSES_FILE = PDF_PARSES_DIR / "sample.jsonl"

# -----------------------------
# Informações do ambiente
# -----------------------------

print("=" * 60)
print("Projeto configurado com sucesso!")
print("=" * 60)

print(f"Modelo: {MODEL_NAME}")
print(f"PyTorch: {torch.__version__}")
print(f"GPU disponível: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("Treinamento será realizado na CPU.")

print("\nEstrutura do projeto:")
print(f"Projeto      : {PROJECT_DIR}")
print(f"Dataset      : {DATASET_DIR}")
print(f"Metadata     : {METADATA_FILE}")
print(f"PDF Parses   : {PDF_PARSES_FILE}")
print(f"Modelos      : {MODEL_DIR}")
print(f"Saídas       : {OUTPUT_DIR}")

print("\nArquivos encontrados:")
print(f"Metadata: {METADATA_FILE.exists()}")
print(f"PDF Parse: {PDF_PARSES_FILE.exists()}")

Projeto configurado com sucesso!
Modelo: google/flan-t5-base
PyTorch: 2.11.0+cu128
GPU disponível: True
Dispositivo: Tesla T4
CUDA: 12.8

Estrutura do projeto:
Projeto      : /content/drive/MyDrive/Colab Notebooks/SubTechChallenge3
Dataset      : /content/drive/MyDrive/Colab Notebooks/SubTechChallenge3/dataset
Metadata     : /content/drive/MyDrive/Colab Notebooks/SubTechChallenge3/dataset/metadata/sample.jsonl
PDF Parses   : /content/drive/MyDrive/Colab Notebooks/SubTechChallenge3/dataset/pdf_parses/sample.jsonl
Modelos      : /content/drive/MyDrive/Colab Notebooks/SubTechChallenge3/model
Saídas       : /content/drive/MyDrive/Colab Notebooks/SubTechChallenge3/output

Arquivos encontrados:
Metadata: True
PDF Parse: True


Etapa 3 – Carregamento e Preparação do Dataset S2ORC

Nesta etapa é realizado o carregamento do conjunto de dados utilizado no treinamento do modelo. Inicialmente, a proposta era utilizar a API oficial do Semantic Scholar para obter o dataset S2ORC (Semantic Scholar Open Research Corpus). Entretanto, durante o desenvolvimento do projeto foi constatado que o acesso ao conjunto completo de dados exige uma API Key, impossibilitando sua utilização dentro do prazo disponível para a execução do trabalho.

Como alternativa, foi utilizado o dataset de exemplo (sample.jsonl) disponibilizado oficialmente pelos mantenedores do projeto S2ORC. Esse conjunto contém a amostra representativa dos dados originais, preservando a mesma estrutura utilizada pelo corpus completo, permitindo reproduzir todas as etapas de preparação, processamento e treinamento sem alterações significativas na metodologia.

Os arquivos utilizados são:

metadata/sample.jsonl: contém informações bibliográficas dos artigos, como título, autores, ano de publicação e demais metadados;
pdf_parses/sample.jsonl: contém o conteúdo textual extraído dos artigos científicos.

Após o carregamento, ambos os arquivos são integrados utilizando o identificador único paper_id, formando um único conjunto de dados que será utilizado nas etapas posteriores de limpeza, preparação dos exemplos de treinamento e ajuste fino (fine-tuning) do modelo FLAN-T5.

In [3]:
# ==========================================================
# ETAPA 3 - Carregamento do Dataset S2ORC
# ==========================================================

import json

# -----------------------------
# Leitura dos arquivos JSONL
# -----------------------------

metadata = pd.read_json(METADATA_FILE, lines=True)
pdf_parses = pd.read_json(PDF_PARSES_FILE, lines=True)

print("=" * 60)
print("Arquivos carregados com sucesso!")
print("=" * 60)

print(f"Metadata: {len(metadata):,} registros")
print(f"PDF Parses: {len(pdf_parses):,} registros")

# -----------------------------
# Junção dos datasets
# -----------------------------

dataset = metadata.merge(
    pdf_parses,
    on="paper_id",
    how="inner",
    suffixes=("_meta", "_pdf")
)

print(f"\nApós o merge: {len(dataset):,} artigos")

# -----------------------------
# Exibir colunas disponíveis
# -----------------------------

print("\nColunas disponíveis:\n")
print(dataset.columns.tolist())

# -----------------------------
# Primeiras linhas
# -----------------------------

dataset.head()

Arquivos carregados com sucesso!
Metadata: 1,000 registros
PDF Parses: 203 registros

Após o merge: 203 artigos

Colunas disponíveis:

['paper_id', 'title', 'authors', 'abstract_meta', 'year', 'arxiv_id', 'acl_id', 'pmc_id', 'pubmed_id', 'doi', 'venue', 'journal', 'mag_id', 'mag_field_of_study', 'outbound_citations', 'inbound_citations', 'has_outbound_citations', 'has_inbound_citations', 'has_pdf_parse', 's2_url', 'has_pdf_body_text', 'has_pdf_parsed_abstract', 'has_pdf_parsed_body_text', 'has_pdf_parsed_bib_entries', 'has_pdf_parsed_ref_entries', '_pdf_hash', 'abstract_pdf', 'body_text', 'bib_entries', 'ref_entries']


,paper_id,title,authors,abstract_meta,year,arxiv_id,acl_id,pmc_id,pubmed_id,doi,...,has_pdf_body_text,has_pdf_parsed_abstract,has_pdf_parsed_body_text,has_pdf_parsed_bib_entries,has_pdf_parsed_ref_entries,_pdf_hash,abstract_pdf,body_text,bib_entries,ref_entries
0,77499681,Effects of Teriparatide Administration on Frac...,"[{'first': 'Chul Hyun', 'middle': [], 'last': ...",None,2016.0,None,NaN,None,NaN,10.4055/jkoa.2016.51.3.231,...,1.0,1.0,1.0,1.0,1.0,11f281316fe4638843a83cf559ce4f60aade00f8,"[{'section': 'Abstract', 'text': 'The purpose ...","[{'section': '', 'text': 'Values are presented...",{'BIBREF0': {'title': 'Bone health and osteopo...,{'FIGREF0': {'text': '비스포스포네이트를 장기간 복용한 골다공증 환...
1,94550656,The Approximate Analysis of Nonlinear Behavior...,"[{'first': 'Mehdi', 'middle': [], 'last': 'Bay...",None,2010.0,None,NaN,None,NaN,None,...,0.0,0.0,0.0,1.0,0.0,42b3e1bd9c4740192f22d8725d470218e86301c8,[],[],{'BIBREF0': {'title': 'Solving ratio-dependent...,{}
2,94551239,Scanning probe memories – Technology and appli...,"[{'first': 'C. David', 'middle': [], 'last': '...",Abstract Scanning probe-based memories have de...,2011.0,None,NaN,None,NaN,10.1016/j.cap.2010.11.130,...,0.0,0.0,0.0,1.0,0.0,b355fc0f19e1945bcb585b0f696da8b01aa4578f,[],[],{'BIBREF2': {'title': 'Optical Near Field Reco...,{}
3,94551546,Gd(III) ion-chelated supramolecular assemblies...,"[{'first': 'Yu', 'middle': [], 'last': 'Zhao',...",An intricate polymer complex can carry genes t...,2015.0,None,NaN,None,NaN,10.1038/am.2015.67,...,1.0,1.0,1.0,1.0,1.0,9bf1cb19041b8ddfca7aeccc9d2f7689c8aa1c7e,"[{'section': 'Abstract', 'text': 'Ethanolamine...","[{'section': 'INTRODUCTION', 'text': 'Gene the...","{'BIBREF0': {'title': 'Cancer statistics', 'au...",{'FIGREF0': {'text': 'General procedures for t...
4,94552339,Analytical Procedure for the Determination of ...,"[{'first': 'Gerald', 'middle': ['W.'], 'last':...",The titanium minerals are essentially oxygenat...,2015.0,None,NaN,None,NaN,None,...,0.0,0.0,0.0,1.0,0.0,7dc1bf397fb5aae2fa07e2697ba6f0237a411bb6,[],[],"{'BIBREF0': {'title': 'Titanium, its occurrenc...",{}


Antes da preparação dos dados, foi realizada a integração dos arquivos metadata/sample.jsonl e pdf_parses/sample.jsonl por meio do campo paper_id. Dos 1.000 registros presentes no arquivo de metadados, 203 possuíam conteúdo textual correspondente, resultando em um conjunto final com 203 artigos completos. Esse conjunto reúne informações como título, resumo, texto do artigo e demais metadados, servindo como base para as próximas etapas de processamento e fine-tuning.

---



Etapa 4 – Análise Exploratória dos Dados

Nesta etapa é realizada uma análise exploratória do conjunto de dados obtido após a integração dos arquivos de metadados e conteúdo textual. O objetivo é compreender as principais características da base, verificar sua qualidade e identificar possíveis inconsistências antes do processo de limpeza e preparação para o treinamento do modelo.

Serão analisadas informações como quantidade de artigos, disponibilidade de resumos e textos completos, distribuição temporal das publicações e tamanho dos documentos, auxiliando na definição dos critérios utilizados nas próximas etapas do projeto.

In [4]:
# ==========================================================
# ETAPA 4 - Análise Exploratória
# ==========================================================

print("=" * 60)
print("Resumo do Dataset")
print("=" * 60)

print(f"Total de artigos: {len(dataset)}")

print(f"\nArtigos com resumo (metadata): {dataset['abstract_meta'].notna().sum()}")
print(f"Artigos com resumo (pdf): {dataset['abstract_pdf'].apply(lambda x: len(x) > 0).sum()}")
print(f"Artigos com corpo do texto: {dataset['body_text'].apply(lambda x: len(x) > 0).sum()}")

print("\nDistribuição por ano:")

display(dataset["year"].value_counts().sort_index().tail(15))

Resumo do Dataset
Total de artigos: 203

Artigos com resumo (metadata): 156
Artigos com resumo (pdf): 78
Artigos com corpo do texto: 92

Distribuição por ano:


,count
year,
2006.0,4
2007.0,8
2008.0,8
2009.0,3
2010.0,11
2011.0,8
2012.0,13
2013.0,16
2014.0,9


Etapa 5 – Limpeza e Preparação dos Dados

Nela vamos:

transformar body_text (lista de seções) em um único texto;
escolher automaticamente o melhor resumo (abstract_pdf ou abstract_meta);
remover artigos sem texto;
criar um DataFrame final pronto para o FLAN-T5, contendo apenas:
paper_id
title
abstract
body

Essa será a base usada para construir os prompts e realizar o fine-tuning.

In [5]:
# ==========================================================
# ETAPA 5 - Limpeza e Preparação dos Dados
# ==========================================================
# Ajuste: o projeto passou a ser de Perguntas e Respostas (Q&A)
# baseado em título + resumo, não mais sumarização do corpo.
# A exigência de "body" não vazio foi removida.

def extract_body(body_sections):
    if not isinstance(body_sections, list):
        return ""
    textos = []
    for section in body_sections:
        if isinstance(section, dict):
            texto = section.get("text", "").strip()
            if texto:
                textos.append(texto)
    return " ".join(textos)


def extract_abstract(row):
    if isinstance(row["abstract_pdf"], list) and len(row["abstract_pdf"]) > 0:
        textos = [
            sec.get("text", "").strip()
            for sec in row["abstract_pdf"]
            if isinstance(sec, dict)
        ]
        textos = [t for t in textos if t]
        if textos:
            return " ".join(textos)
    if pd.notna(row["abstract_meta"]):
        return str(row["abstract_meta"]).strip()
    return ""


dataset["abstract"] = dataset.apply(extract_abstract, axis=1)
dataset["body"] = dataset["body_text"].apply(extract_body)  # mantido só como referência

final_dataset = dataset[["paper_id", "title", "abstract", "body"]].copy()

final_dataset = final_dataset[
    (final_dataset["title"].notna()) &
    (final_dataset["abstract"].str.strip().str.len() > 0)
].reset_index(drop=True)

print("=" * 60)
print("Dataset preparado!")
print("=" * 60)
print(f"Artigos prontos para treinamento: {len(final_dataset)}")

display(final_dataset.head())

Dataset preparado!
Artigos prontos para treinamento: 164


,paper_id,title,abstract,body
0,77499681,Effects of Teriparatide Administration on Frac...,The purpose of this study is to evaluate the e...,Values are presented as number only or median ...
1,94551239,Scanning probe memories – Technology and appli...,Abstract Scanning probe-based memories have de...,
2,94551546,Gd(III) ion-chelated supramolecular assemblies...,Ethanolamine (EA) or ethylenediamine (ED)-func...,Gene therapy holds potential for treating many...
3,94552339,Analytical Procedure for the Determination of ...,The titanium minerals are essentially oxygenat...,
4,94553452,Modelling the atmospheric boundary layer in a ...,"In this study, we describe and validate a boun...",


Após a integração dos arquivos, foram obtidos 203 registros. Entretanto, apenas 87 artigos continham simultaneamente corpo textual e resumo, requisitos necessários para a tarefa de sumarização utilizada no fine-tuning do FLAN-T5. Os demais registros foram descartados por ausência de uma dessas informações.

Etapa 6 – Preparação dos Dados para Fine-Tuning

Nesta etapa, os artigos científicos são convertidos para o formato de entrada esperado pelo modelo FLAN-T5. Como o modelo foi originalmente treinado para seguir instruções (instruction tuning), cada exemplo é estruturado como um par de entrada e saída. A entrada contém uma instrução solicitando o resumo do artigo juntamente com parte do seu conteúdo textual, enquanto a saída corresponde ao resumo do próprio artigo. Esse formato permite que o modelo aprenda a realizar a tarefa de sumarização durante o processo de fine-tuning.

In [6]:
# ==========================================================
# ETAPA 6 - Preparação dos exemplos de Perguntas e Respostas
# ==========================================================

import re

# ----------------------------------------------------------
# Perguntas-modelo (em inglês, mesmo idioma do abstract e do
# prompt) + palavras-chave usadas para localizar a frase do
# resumo que melhor responde cada pergunta
# ----------------------------------------------------------

QUESTION_TEMPLATES = [
    ("What is the objective of this scientific article?",
     ["objective", "aim", "purpose", "this paper", "this study", "we study", "we investigate"]),
    ("What methodology was used in this study?",
     ["method", "approach", "we propose", "we use", "based on", "using"]),
    ("What were the main results found?",
     ["result", "found", "show that", "demonstrate", "achieve", "indicate"]),
    ("What is the conclusion of the study?",
     ["conclusion", "conclude", "in summary", "suggest that", "overall"]),
]

def extract_answer(abstract, keywords):
    sentences = re.split(r'(?<=[.!?])\s+', abstract.strip())
    matched = [s.strip() for s in sentences if any(k in s.lower() for k in keywords)]
    if matched:
        return " ".join(matched)
    return abstract.strip()  # fallback: resumo inteiro

def build_prompt(title, abstract, question):
    return f"""
You are an expert scientific assistant.

Read the title and abstract of the scientific article below and
answer the user's question based only on this information.

Title:
{title}

Abstract:
{abstract}

Question:
{question}

Answer:
""".strip()

# ----------------------------------------------------------
# Construção do dataset de Q&A (várias perguntas por artigo)
# ----------------------------------------------------------

rows = []

for _, row in final_dataset.iterrows():
    abstract = str(row["abstract"]).strip()
    title = row["title"] if pd.notna(row["title"]) else ""

    for question, keywords in QUESTION_TEMPLATES:
        answer = extract_answer(abstract, keywords)
        rows.append({
            "paper_id": row["paper_id"],
            "title": title,
            "question": question,
            "input_text": build_prompt(title, abstract, question),
            "target_text": answer
        })

training_df = pd.DataFrame(rows)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

input_sizes = [
    len(tokenizer(x, add_special_tokens=True)["input_ids"])
    for x in training_df["input_text"]
]

print("=" * 60)
print("PREPARAÇÃO DOS EXEMPLOS (PERGUNTA E RESPOSTA)")
print("=" * 60)
print(f"Artigos originais       : {len(final_dataset)}")
print(f"Perguntas por artigo    : {len(QUESTION_TEMPLATES)}")
print(f"Total de exemplos       : {len(training_df)}")
print(f"Média de tokens (input) : {sum(input_sizes)/len(input_sizes):.1f}")
print(f"Maior entrada           : {max(input_sizes)}")
print(f"Menor entrada           : {min(input_sizes)}")

display(training_df.head(8))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1146 > 512). Running this sequence through the model will result in indexing errors


PREPARAÇÃO DOS EXEMPLOS (PERGUNTA E RESPOSTA)
Artigos originais       : 164
Perguntas por artigo    : 4
Total de exemplos       : 656
Média de tokens (input) : 340.9
Maior entrada           : 1479
Menor entrada           : 66


,paper_id,title,question,input_text,target_text
0,77499681,Effects of Teriparatide Administration on Frac...,What is the objective of this scientific article?,You are an expert scientific assistant.\n\nRea...,The purpose of this study is to evaluate the e...
1,77499681,Effects of Teriparatide Administration on Frac...,What methodology was used in this study?,You are an expert scientific assistant.\n\nRea...,Materials and Methods: We retrospectively revi...
2,77499681,Effects of Teriparatide Administration on Frac...,What were the main results found?,You are an expert scientific assistant.\n\nRea...,Clinical results were assessed using the Nakaj...
3,77499681,Effects of Teriparatide Administration on Frac...,What is the conclusion of the study?,You are an expert scientific assistant.\n\nRea...,Conclusion: The injection group showed better ...
4,94551239,Scanning probe memories – Technology and appli...,What is the objective of this scientific article?,You are an expert scientific assistant.\n\nRea...,In this paper we discuss the three major famil...
5,94551239,Scanning probe memories – Technology and appli...,What methodology was used in this study?,You are an expert scientific assistant.\n\nRea...,In this paper we discuss the three major famil...
6,94551239,Scanning probe memories – Technology and appli...,What were the main results found?,You are an expert scientific assistant.\n\nRea...,Abstract Scanning probe-based memories have de...
7,94551239,Scanning probe memories – Technology and appli...,What is the conclusion of the study?,You are an expert scientific assistant.\n\nRea...,Abstract Scanning probe-based memories have de...


6.1 – Tokenização e Divisão do Dataset

Antes do treinamento, os textos precisam ser convertidos em tokens compreensíveis pelo modelo FLAN-T5. Nesta etapa é carregado o tokenizer oficial, realizada a tokenização das entradas e saídas e, posteriormente, o conjunto de dados é dividido em subconjuntos de treinamento e validação. Essa divisão permite acompanhar o desempenho do modelo durante o fine-tuning e verificar sua capacidade de generalização.

In [7]:
# ==========================================================
# ETAPA 6.1 - Tokenização e divisão do dataset
# ==========================================================

from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer
import pandas as pd

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

input_lengths = [len(tokenizer(text).input_ids) for text in training_df["input_text"]]
target_lengths = [len(tokenizer(text).input_ids) for text in training_df["target_text"]]

stats = pd.DataFrame({"input_tokens": input_lengths, "target_tokens": target_lengths})

print("=" * 60)
print("ESTATÍSTICAS DOS DADOS")
print("=" * 60)
display(stats.describe())

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128  # respostas são trechos do abstract, não o resumo inteiro

truncated = sum(x > MAX_INPUT_LENGTH for x in input_lengths)
print(f"\nExemplos truncados: {truncated}/{len(input_lengths)}")
print(f"Percentual: {100*truncated/len(input_lengths):.1f}%")

train_df, valid_df = train_test_split(
    training_df, test_size=0.20, random_state=SEED, shuffle=True
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
valid_dataset = Dataset.from_pandas(valid_df.reset_index(drop=True))

def preprocess(example):
    model_inputs = tokenizer(example["input_text"], max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=example["target_text"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(preprocess, remove_columns=train_dataset.column_names)
valid_dataset = valid_dataset.map(preprocess, remove_columns=valid_dataset.column_names)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
valid_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("=" * 60)
print("TOKENIZAÇÃO CONCLUÍDA")
print("=" * 60)
print(f"Treino: {len(train_dataset)}")
print(f"Validação: {len(valid_dataset)}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1146 > 512). Running this sequence through the model will result in indexing errors


ESTATÍSTICAS DOS DADOS


,input_tokens,target_tokens
count,656.000000,656.000000
mean,340.914634,178.461890
std,202.034489,178.144384
min,66.000000,2.000000
25%,226.250000,54.000000
50%,305.500000,122.000000
75%,406.000000,247.000000
max,1479.000000,1416.000000



Exemplos truncados: 80/656
Percentual: 12.2%


Map:   0%|          | 0/524 [00:00<?, ? examples/s]

Map:   0%|          | 0/132 [00:00<?, ? examples/s]

TOKENIZAÇÃO CONCLUÍDA
Treino: 524
Validação: 132


6.2 – Carregamento do Modelo Fundacional

Nesta etapa é carregado o modelo fundacional FLAN-T5 Small, que servirá como base para o processo de fine-tuning. Inicialmente, o modelo é utilizado em seu estado original, ou seja, apenas com os conhecimentos adquiridos durante seu pré-treinamento. Esse carregamento permite realizar testes iniciais antes do treinamento e, posteriormente, comparar seu desempenho com a versão ajustada utilizando os artigos científicos selecionados.

In [8]:
# ==========================================================
# ETAPA 6.2 - Carregamento do Modelo Fundacional
# ==========================================================

from transformers import AutoModelForSeq2SeqLM

# Carrega o modelo pré-treinado
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Move para GPU, caso disponível
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("=" * 60)
print("Modelo carregado com sucesso!")
print("=" * 60)

print(f"Modelo: {MODEL_NAME}")
print(f"Dispositivo: {device}")
print(f"Número de parâmetros: {model.num_parameters():,}")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Modelo carregado com sucesso!
Modelo: google/flan-t5-base
Dispositivo: cuda
Número de parâmetros: 247,577,856


6.3 – Teste Inicial do Modelo

Nesta etapa são realizados testes utilizando o modelo fundacional em seu estado original, antes da execução do fine-tuning. Para isso, são utilizadas perguntas relacionadas aos artigos científicos presentes no conjunto de dados preparado anteriormente. As respostas obtidas servirão como referência para comparação com os resultados gerados após o treinamento do modelo, permitindo avaliar a eficácia do processo de fine-tuning.

In [9]:
# ==========================================================
# ETAPA 6.3 - Teste Inicial do Modelo
# ==========================================================

import torch

model.eval()

test_examples = training_df.sample(n=3, random_state=42).reset_index(drop=True)

print("=" * 100)
print("TESTE INICIAL DO MODELO (ANTES DO FINE-TUNING)")
print("=" * 100)

for i, example in test_examples.iterrows():
    inputs = tokenizer(
        example["input_text"], return_tensors="pt", truncation=True, max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, num_beams=4, early_stopping=True)

    resposta_modelo = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"\nARTIGO {i+1}")
    print("=" * 100)
    print(f"Título: {example['title']}\n")
    print("PERGUNTA")
    print(example["question"])
    print("\nRESPOSTA DO MODELO")
    print(resposta_modelo)
    print("\nRESPOSTA ESPERADA")
    print(example["target_text"])
    print("\n" + "=" * 100)

TESTE INICIAL DO MODELO (ANTES DO FINE-TUNING)

ARTIGO 1
Título: E2E: embracing user heterogeneity to improve quality of experience on the web

PERGUNTA
What is the objective of this scientific article?

RESPOSTA DO MODELO
to improve QoE

RESPOSTA ESPERADA
This paper presents E2E, the first resource allocation system that embraces user heterogeneity to allocate server-side resources in a QoE-aware manner.


ARTIGO 2
Título: Complete genomic sequence of bacteriophage P23: a novel Vibrio phage isolated from the Yellow Sea, China

PERGUNTA
What methodology was used in this study?

RESPOSTA DO MODELO
Transmission electron microscopy

RESPOSTA ESPERADA
A novel Vibrio phage, P23, belonging to the family Siphoviridae was isolated from the surface water of the Yellow Sea, China. The complete genome of this phage was determined. A one-step growth curve showed that the latent period was approximately 30 min, the burst size was 24 PFU/cell, and the rise period was 20 min. The phage is host specif

7.1 – Configuração do Fine-Tuning

Nesta etapa são definidos os parâmetros utilizados no processo de fine-tuning do modelo fundacional. São configurados os hiperparâmetros de treinamento, como número de épocas, tamanho dos lotes, taxa de aprendizado e estratégia de avaliação. Essas configurações orientam o processo de ajuste dos pesos do modelo, permitindo que ele aprenda a gerar resumos mais adequados ao conjunto de artigos científicos selecionado.

In [10]:
# ==========================================================
# ETAPA 7.1 - Configuração do Fine-Tuning
# ==========================================================

from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq
from torch.optim import AdamW

# -----------------------------
# Hiperparâmetros
# -----------------------------

EPOCHS = 5
LEARNING_RATE = 1e-4
BATCH_SIZE = 1

# -----------------------------
# Data Collator
# -----------------------------

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# -----------------------------
# DataLoaders
# -----------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

# -----------------------------
# Otimizador
# -----------------------------

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

device = model.device

# -----------------------------
# Histórico
# -----------------------------

train_history = []
valid_history = []

print("=" * 60)
print("CONFIGURAÇÃO DO FINE-TUNING")
print("=" * 60)

print(f"Épocas................: {EPOCHS}")
print(f"Learning Rate.........: {LEARNING_RATE}")
print(f"Batch treino..........: {BATCH_SIZE}")
print(f"Batch validação.......: {BATCH_SIZE}")
print(f"Treino................: {len(train_dataset)} exemplos")
print(f"Validação.............: {len(valid_dataset)} exemplos")

CONFIGURAÇÃO DO FINE-TUNING
Épocas................: 5
Learning Rate.........: 0.0001
Batch treino..........: 1
Batch validação.......: 1
Treino................: 524 exemplos
Validação.............: 132 exemplos


7.2 – Execução do Fine-Tuning

Nesta etapa é iniciado o processo de fine-tuning do modelo fundacional utilizando o conjunto de treinamento preparado anteriormente. Durante o treinamento, o modelo ajusta seus parâmetros com base nos exemplos fornecidos, enquanto o conjunto de validação é utilizado para acompanhar seu desempenho ao final de cada época. Ao término do processo, são apresentados os principais resultados do treinamento.

In [11]:
!pip install -U datasets --quiet

In [12]:
# ==========================================================
# ETAPA 7.2 - Fine-Tuning Manual
# ==========================================================

import math
import time

from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq
from torch.optim import AdamW

print("=" * 60)
print("INICIANDO O FINE-TUNING MANUAL")
print("=" * 60)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collator
)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

device = model.device

epochs = 5

inicio = time.time()

for epoch in range(epochs):

    print(f"\n================ ÉPOCA {epoch+1}/{epochs} ================\n")

    model.train()

    total_loss = 0

    for batch_idx, batch in enumerate(train_loader):

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)

        loss = outputs.loss

        print(f"Batch {batch_idx:03d} | Loss = {loss.item()}")

        if torch.isnan(loss):

            print("\n\n*************")
            print("NAN ENCONTRADO")
            print("*************")

            print("Batch:", batch_idx)

            print("input_ids")
            print(batch["input_ids"])

            print("labels")
            print(batch["labels"])

            raise RuntimeError("Treinamento interrompido por NaN.")

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

    media = total_loss / len(train_loader)

    print(f"\nLoss média treino = {media:.4f}")

    # -----------------------------
    # Validação
    # -----------------------------

    model.eval()

    valid_loss = 0

    with torch.no_grad():

        for batch_idx, batch in enumerate(valid_loader):

            batch = {
                k: v.to(device)
                for k, v in batch.items()
            }

            outputs = model(**batch)

            loss = outputs.loss

            if torch.isnan(loss):

                print("\nNaN na validação.")
                print("Batch:", batch_idx)

                raise RuntimeError("NaN na validação.")

            valid_loss += loss.item()

    valid_loss /= len(valid_loader)

    print(f"Loss validação = {valid_loss:.4f}")

fim = time.time()

print("\n")
print("=" * 60)
print("TREINAMENTO FINALIZADO")
print("=" * 60)

print(f"Tempo: {(fim-inicio)/60:.2f} minutos")

INICIANDO O FINE-TUNING MANUAL

================ ÉPOCA 1/5 ================

Batch 000 | Loss = 0.1714015007019043


/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Batch 001 | Loss = 0.2678943872451782
Batch 002 | Loss = 0.38946476578712463
Batch 003 | Loss = 0.5596798062324524
Batch 004 | Loss = 0.33173131942749023
Batch 005 | Loss = 0.12780462205410004
Batch 006 | Loss = 0.3082859218120575
Batch 007 | Loss = 0.373590886592865
Batch 008 | Loss = 0.1515274941921234
Batch 009 | Loss = 0.3064280152320862
Batch 010 | Loss = 2.6587088108062744
Batch 011 | Loss = 0.5275428295135498
Batch 012 | Loss = 0.24328915774822235
Batch 013 | Loss = 0.16616584360599518
Batch 014 | Loss = 0.2452322393655777
Batch 015 | Loss = 0.2259007841348648
Batch 016 | Loss = 0.15130235254764557
Batch 017 | Loss = 0.26783356070518494
Batch 018 | Loss = 0.07059600204229355
Batch 019 | Loss = 0.14027370512485504
Batch 020 | Loss = 0.3492691218852997
Batch 021 | Loss = 0.26743006706237793
Batch 022 | Loss = 0.2537207007408142
Batch 023 | Loss = 0.19884027540683746
Batch 024 | Loss = 0.4247262477874756
Batch 025 | Loss = 0.16647818684577942
Batch 026 | Loss = 0.083585225045681
Ba

ETAPA 7.3 – Salvamento do Modelo Treinado
Objetivo

Após o término do fine-tuning, o modelo ajustado é salvo em disco para que possa ser reutilizado posteriormente sem a necessidade de repetir todo o treinamento. Também é salvo o tokenizer utilizado durante o treinamento, garantindo que a tokenização durante a inferência seja idêntica à utilizada no ajuste fino.

In [13]:
# ==========================================================
# ETAPA 7.3 - Salvando o modelo treinado
# ==========================================================

MODEL_OUTPUT_DIR = PROJECT_DIR / "modelo_finetunado"

MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print("=" * 60)
print("MODELO SALVO COM SUCESSO")
print("=" * 60)
print(f"Local: {MODEL_OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MODELO SALVO COM SUCESSO
Local: /content/drive/MyDrive/Colab Notebooks/SubTechChallenge3/modelo_finetunado


ETAPA 8 – Avaliação do Modelo Fine-Tunado
Objetivo

Nesta etapa será realizada a avaliação qualitativa do modelo após o fine-tuning. Para isso, o modelo treinado é carregado e utilizado para gerar resumos de artigos científicos presentes no conjunto de validação. Os resultados produzidos pelo modelo são comparados com os resumos originais do dataset, permitindo verificar a qualidade do ajuste fino realizado.

In [14]:
# ==========================================================
# ETAPA 8 - Avaliação do modelo (Perguntas e Respostas)
# ==========================================================

from rouge_score import rouge_scorer, scoring
from transformers import AutoModelForSeq2SeqLM
import numpy as np
import pandas as pd
import torch

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def generate_answers(gen_model):
    gen_model.eval()
    preds = []
    for sample in valid_dataset:
        input_ids = sample["input_ids"].unsqueeze(0).to(device)
        attention_mask = sample["attention_mask"].unsqueeze(0).to(device)
        with torch.no_grad():
            outputs = gen_model.generate(
                input_ids=input_ids, attention_mask=attention_mask,
                max_new_tokens=MAX_TARGET_LENGTH, num_beams=4, early_stopping=True,
                repetition_penalty=2.0, no_repeat_ngram_size=3
            )
        preds.append(tokenizer.decode(outputs[0], skip_special_tokens=True))
    return preds

references = [tokenizer.decode(s["labels"], skip_special_tokens=True) for s in valid_dataset]
questions = valid_df["question"].reset_index(drop=True)

print("Gerando respostas com o modelo FINE-TUNADO...")
finetuned_preds = generate_answers(model)

print("Gerando respostas com o modelo ORIGINAL (antes do fine-tuning)...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
baseline_preds = generate_answers(base_model)
del base_model
torch.cuda.empty_cache()

def score_answers(preds, refs):
    aggregator = scoring.BootstrapAggregator()
    per_example = []
    for ref, pred in zip(refs, preds):
        scores = scorer.score(ref, pred)
        aggregator.add_scores(scores)
        per_example.append({m: scores[m].fmeasure for m in ["rouge1", "rouge2", "rougeL"]})
    return aggregator.aggregate(), pd.DataFrame(per_example)

result_ft, df_ft = score_answers(finetuned_preds, references)
result_base, df_base = score_answers(baseline_preds, references)

rows = []
for metric in ["rouge1", "rouge2", "rougeL"]:
    antes = result_base[metric].mid.fmeasure
    depois = result_ft[metric].mid.fmeasure
    rows.append({
        "Métrica": metric.upper(),
        "Antes (baseline)": round(antes, 4),
        "Depois (fine-tuned)": round(depois, 4),
        "Delta": round(depois - antes, 4),
        "Melhoria %": f"{(depois - antes) / antes * 100:+.1f}%" if antes > 0 else "n/a"
    })

print("=" * 80)
print("COMPARAÇÃO: ANTES x DEPOIS DO FINE-TUNING")
print("=" * 80)
display(pd.DataFrame(rows))

print("\n" + "=" * 80)
print("EXEMPLOS — ANTES x DEPOIS")
print("=" * 80)

for i in range(min(5, len(valid_dataset))):
    print(f"\n{'-'*80}\nEXEMPLO {i+1}\n{'-'*80}")
    print(f"[PERGUNTA]\n{questions[i]}\n")
    print(f"[RESPOSTA ESPERADA]\n{references[i][:300]}\n")
    print(f"[ANTES do fine-tuning]  (ROUGE-1: {df_base['rouge1'][i]:.3f})\n{baseline_preds[i][:300]}\n")
    print(f"[DEPOIS do fine-tuning] (ROUGE-1: {df_ft['rouge1'][i]:.3f})\n{finetuned_preds[i][:300]}")

Gerando respostas com o modelo FINE-TUNADO...
Gerando respostas com o modelo ORIGINAL (antes do fine-tuning)...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


COMPARAÇÃO: ANTES x DEPOIS DO FINE-TUNING


,Métrica,Antes (baseline),Depois (fine-tuned),Delta,Melhoria %
0,ROUGE1,0.2234,0.6231,0.3996,+178.8%
1,ROUGE2,0.1641,0.5442,0.3801,+231.6%
2,ROUGEL,0.2065,0.5972,0.3907,+189.2%



EXEMPLOS — ANTES x DEPOIS

--------------------------------------------------------------------------------
EXEMPLO 1
--------------------------------------------------------------------------------
[PERGUNTA]
What is the objective of this scientific article?

[RESPOSTA ESPERADA]
This paper presents E2E, the first resource allocation system that embraces user heterogeneity to allocate server-side resources in a QoE-aware manner.

[ANTES do fine-tuning]  (ROUGE-1: 0.154)
to improve QoE

[DEPOIS do fine-tuning] (ROUGE-1: 1.000)
This paper presents E2E, the first resource allocation system that embraces user heterogeneity to allocate server-side resources in a QoE-aware manner.

--------------------------------------------------------------------------------
EXEMPLO 2
--------------------------------------------------------------------------------
[PERGUNTA]
What methodology was used in this study?

[RESPOSTA ESPERADA]
A novel Vibrio phage, P23, belonging to the family Siphoviridae was i

Etapa 9 – Geração de Respostas a partir de Perguntas do Usuário

Objetivo

Nesta etapa final, o modelo fine-tunado é utilizado em seu propósito real: responder perguntas feitas livremente pelo usuário sobre um artigo científico específico do dataset. É apresentada uma lista de artigos disponíveis, o usuário escolhe um pelo índice e digita sua pergunta, e o modelo gera uma resposta com base no título e resumo do artigo selecionado, seguindo o mesmo formato de prompt utilizado durante o treinamento.

Diferente das perguntas fixas usadas na Etapa 6 (geração dos exemplos de treino) e na Etapa 8 (avaliação), aqui a pergunta é livre — o que permite avaliar a capacidade de generalização do modelo para perguntas nunca vistas durante o fine-tuning.

Observação: como todo o treinamento foi realizado em inglês (título, resumo e perguntas), a pergunta do usuário deve ser feita em inglês para manter consistência com o padrão aprendido pelo modelo.

In [15]:
# ==========================================================
# ETAPA 9 - Geração de Respostas a partir de Perguntas do Usuário
# ==========================================================
# Objetivo: permitir que o usuário escolha um artigo do dataset
# e faça uma pergunta livre sobre ele, usando o modelo já
# fine-tunado (carregado ao final da ETAPA 7.3 / avaliado na 8).

# ----------------------------------------------------------
# Lista de artigos disponíveis para consulta
# ----------------------------------------------------------

print("=" * 80)
print("ARTIGOS DISPONÍVEIS")
print("=" * 80)

for i, row in final_dataset.head(20).iterrows():
    print(f"[{i}] {row['title']}")

print(f"\n(mostrando 20 de {len(final_dataset)} artigos)")

# ----------------------------------------------------------
# Função de resposta
# ----------------------------------------------------------

def answer_question(article_index, question):

    article = final_dataset.loc[article_index]

    prompt = build_prompt(
        article["title"],
        article["abstract"],
        question
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    ).to(device)

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=4,
            early_stopping=True,
            repetition_penalty=2.0,
            no_repeat_ngram_size=3
        )

    resposta = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("=" * 80)
    print(f"Artigo : {article['title']}")
    print(f"Pergunta: {question}")
    print("-" * 80)
    print(f"Resposta do modelo:\n{resposta}")
    print("=" * 80)

    return resposta

# ----------------------------------------------------------
# Entrada interativa do usuário
# ----------------------------------------------------------

article_index = int(input("Digite o índice do artigo (veja a lista acima): "))
user_question = input("Digite sua pergunta (em inglês, para manter consistência com o treino): ")

answer_question(article_index, user_question)

ARTIGOS DISPONÍVEIS
[0] Effects of Teriparatide Administration on Fracture Healing after Intramedullary Nailing in Atypical Femoral Fractures
[1] Scanning probe memories – Technology and applications
[2] Gd(III) ion-chelated supramolecular assemblies composed of PGMA-based polycations for effective biomedical applications
[3] Analytical Procedure for the Determination of Titanium and Iron in Titaniferous Ores
[4] Modelling the atmospheric boundary layer in a climate model of intermediate complexity
[5] Catalysis without precious metals
[6] Biodegradation of Chlorinated Solvents: Reactions near DNAPL and Enzyme Function
[7] No. 8 Cultural and other relations - Mapping Central Asia’s relations with other Asian states
[8] The Ship Owner's Lien on Sub-freights and Personal Property Securities Regimes
[9] An update on model Ayush wellness clinic at president’s estate, India
[10] Distributionally Robust Counterpart in Markov Decision Processes
[11] Adult and larval photoreceptors use differe

'Preface CATALYSIS INVOLVING THE H*TRANSFER REACTIONS OF FIRST-ROW TRANSITION METALS H*Transfer Between M-H Bonds and Organic Radicals H*transfer Between Ligands and organic Radical S&C Bonds Chain Transfer CatalysisCatalysis of Radical Cyclizations Competing Methods for the Cyclingization of Dienes Summary and Conclusions CATALLYYTIC REDUCTION OF DINITROGEN TO AMMONIA BY MOLYBDENUM Introduction Some Characteristics of Triamid'